# Extraction Quality Comparison

This notebook compares all extraction plans (Baseline, Plan A, B, C, D) on identical test data to determine the best approach for production.

## Test Data
- Nexus AI: Perfect mock data with all fields explicitly stated
- Zinnia: Perfect mock data for consistency validation

## Metrics Tracked
- Confidence scores per extraction type
- Validation error count and types
- Cost per memo (tokens, API calls, $)
- Processing time
- Field extraction accuracy


In [ ]:
import sys
import os
import asyncio
import json
import time
from pathlib import Path
from typing import Dict, List, Any
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

from app.services.document_parser import LangChainDocumentParser
from app.services.semantic_router import SemanticRouter
from app.services.extractors import ExtractionCoordinator
from app.services.extractors_plan_a import ExtractorPlanA
from app.services.extractors_plan_b import ExtractorPlanB
from app.services.extractors_plan_c import ExtractorPlanC
from app.services.extractors_plan_d import ExtractorPlanD
from app.core.models import DEFAULT_TEMPLATE

# Set up plotting
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


## Helper Functions


In [ ]:
def load_test_documents(company_name: str) -> List[Dict[str, Any]]:
    """Load test documents for a company"""
    base_path = Path.cwd().parent / "mock-data" / company_name.lower().replace(" ", "-")
    
    documents = []
    for file_path in base_path.glob("*.docx"):
        with open(file_path, "rb") as f:
            content = f.read()
            documents.append({
                "filename": file_path.name,
                "content": content,
                "content_type": "application/vnd.openxmlformats-officedocument.wordprocessingml.document"
            })
    
    for file_path in base_path.glob("*.xlsx"):
        with open(file_path, "rb") as f:
            content = f.read()
            documents.append({
                "filename": file_path.name,
                "content": content,
                "content_type": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
            })
    
    return documents


async def run_extraction_plan(
    plan_name: str,
    extractor,
    routed_chunks: Dict[str, List],
    extraction_types: List[str]
) -> Dict[str, Any]:
    """Run extraction using a specific plan"""
    print(f"\n{'='*70}")
    print(f"Testing {plan_name}")
    print(f"{'='*70}")
    
    start_time = time.time()
    results = {}
    total_cost = 0.0
    validation_errors = []
    
    for extract_type in extraction_types:
        chunks = routed_chunks.get(extract_type, [])
        if not chunks:
            continue
        
        try:
            extracted_data = await extractor.extract(chunks, extract_type)
            results[extract_type] = extracted_data
            
            # Track stats
            if hasattr(extracted_data, "_extraction_stats"):
                stats = extracted_data._extraction_stats
                total_cost += stats.get("cost", 0.0)
            
            # Check for validation errors
            if hasattr(extracted_data, "uncertainty_flags"):
                for flag in extracted_data.uncertainty_flags:
                    if "validation" in flag.lower() or "error" in flag.lower():
                        validation_errors.append(f"{extract_type}: {flag}")
        
        except Exception as e:
            print(f"  ✗ {extract_type} extraction failed: {e}")
            validation_errors.append(f"{extract_type}: {str(e)}")
    
    elapsed = time.time() - start_time
    
    # Extract confidence scores
    confidence_scores = {}
    for extract_type, data in results.items():
        if hasattr(data, "confidence"):
            confidence_scores[extract_type] = data.confidence
    
    return {
        "plan_name": plan_name,
        "results": results,
        "confidence_scores": confidence_scores,
        "validation_errors": validation_errors,
        "total_cost": total_cost,
        "processing_time": elapsed,
    }


async def run_full_pipeline_test(
    company_name: str,
    funding_stage: str,
    plan_name: str,
    extractor_class
) -> Dict[str, Any]:
    """Run full pipeline test with a specific extractor plan"""
    
    # Load documents
    documents = load_test_documents(company_name)
    print(f"\nLoaded {len(documents)} documents for {company_name}")
    
    # Parse and chunk
    parser = LangChainDocumentParser()
    chunks = await parser.parse_documents(documents)
    print(f"Generated {len(chunks)} chunks")
    
    # Route chunks
    router = SemanticRouter()
    routed_chunks = await router.route_chunks(chunks)
    
    # Create extractor
    extractor = extractor_class()
    
    # Run extractions
    extraction_types = ["progress", "financial", "market", "company", "team"]
    result = await run_extraction_plan(plan_name, extractor, routed_chunks, extraction_types)
    
    return result


## Run All Plans on Nexus AI Data


In [ ]:
# Test all plans on Nexus AI
nexus_results = {}

# Baseline - need to create a wrapper class
class BaselineExtractorWrapper:
    def __init__(self):
        from app.services.extractors import ExtractionCoordinator
        self.coordinator = ExtractionCoordinator()
        self.extractor = self.coordinator.extractor
    
    async def extract(self, documents, extract_type):
        return await self.extractor.extract(documents, extract_type)

nexus_results["Baseline"] = await run_full_pipeline_test(
    "Nexus AI", "Series A", "Baseline", BaselineExtractorWrapper
)

# Plan A
nexus_results["Plan A"] = await run_full_pipeline_test(
    "Nexus AI", "Series A", "Plan A", ExtractorPlanA
)

# Plan B
nexus_results["Plan B"] = await run_full_pipeline_test(
    "Nexus AI", "Series A", "Plan B", ExtractorPlanB
)

# Plan C (if Claude available)
try:
    nexus_results["Plan C"] = await run_full_pipeline_test(
        "Nexus AI", "Series A", "Plan C", ExtractorPlanC
    )
except Exception as e:
    print(f"Plan C skipped: {e}")

# Plan D
nexus_results["Plan D"] = await run_full_pipeline_test(
    "Nexus AI", "Series A", "Plan D", ExtractorPlanD
)


## Compare Results


In [ ]:
# Create comparison DataFrame
comparison_data = []

for plan_name, result in nexus_results.items():
    confidence_scores = result["confidence_scores"]
    
    comparison_data.append({
        "Plan": plan_name,
        "Progress Confidence": confidence_scores.get("progress", 0.0),
        "Financial Confidence": confidence_scores.get("financial", 0.0),
        "Market Confidence": confidence_scores.get("market", 0.0),
        "Company Confidence": confidence_scores.get("company", 0.0),
        "Team Confidence": confidence_scores.get("team", 0.0),
        "Avg Confidence": sum(confidence_scores.values()) / len(confidence_scores) if confidence_scores else 0.0,
        "Validation Errors": len(result["validation_errors"]),
        "Cost ($)": result["total_cost"],
        "Time (s)": result["processing_time"],
    })

df = pd.DataFrame(comparison_data)
print("\n" + "="*70)
print("COMPARISON RESULTS")
print("="*70)
print(df.to_string(index=False))


## Visualizations


In [ ]:
# Confidence scores comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Average confidence
axes[0, 0].bar(df["Plan"], df["Avg Confidence"], color='steelblue')
axes[0, 0].set_title("Average Confidence Score")
axes[0, 0].set_ylabel("Confidence")
axes[0, 0].set_ylim(0, 1.0)
axes[0, 0].axhline(y=0.8, color='r', linestyle='--', label='Target (0.8)')
axes[0, 0].legend()

# Cost comparison
axes[0, 1].bar(df["Plan"], df["Cost ($)"], color='orange')
axes[0, 1].set_title("Cost per Memo")
axes[0, 1].set_ylabel("Cost ($)")

# Validation errors
axes[1, 0].bar(df["Plan"], df["Validation Errors"], color='red')
axes[1, 0].set_title("Validation Errors")
axes[1, 0].set_ylabel("Error Count")

# Processing time
axes[1, 1].bar(df["Plan"], df["Time (s)"], color='green')
axes[1, 1].set_title("Processing Time")
axes[1, 1].set_ylabel("Time (seconds)")

plt.tight_layout()
plt.show()

# Detailed confidence by extraction type
fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(df))
width = 0.15

ax.bar([i - 2*width for i in x], df["Progress Confidence"], width, label='Progress')
ax.bar([i - width for i in x], df["Financial Confidence"], width, label='Financial')
ax.bar(x, df["Market Confidence"], width, label='Market')
ax.bar([i + width for i in x], df["Company Confidence"], width, label='Company')
ax.bar([i + 2*width for i in x], df["Team Confidence"], width, label='Team')

ax.set_xlabel('Plan')
ax.set_ylabel('Confidence Score')
ax.set_title('Confidence Scores by Extraction Type')
ax.set_xticks(x)
ax.set_xticklabels(df["Plan"])
ax.legend()
ax.set_ylim(0, 1.0)
ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target (0.8)')

plt.tight_layout()
plt.show()


## Quality vs Cost Analysis


In [ ]:
# Scatter plot: Quality vs Cost
fig, ax = plt.subplots(figsize=(10, 6))

for idx, row in df.iterrows():
    ax.scatter(row["Cost ($)"], row["Avg Confidence"], s=200, alpha=0.6)
    ax.annotate(row["Plan"], (row["Cost ($)"], row["Avg Confidence"]), 
                xytext=(5, 5), textcoords='offset points')

ax.set_xlabel('Cost per Memo ($)')
ax.set_ylabel('Average Confidence Score')
ax.set_title('Quality vs Cost Trade-off')
ax.axhline(y=0.8, color='r', linestyle='--', alpha=0.5, label='Target Confidence (0.8)')
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

# Print recommendation
print("\n" + "="*70)
print("RECOMMENDATION")
print("="*70)

# Find plans that meet confidence target
meets_target = df[df["Avg Confidence"] >= 0.8]
if len(meets_target) > 0:
    # Recommend the one with lowest cost among those meeting target
    best = meets_target.loc[meets_target["Cost ($)"].idxmin()]
    print(f"\n✅ Best Plan: {best['Plan']}")
    print(f"   Average Confidence: {best['Avg Confidence']:.2f}")
    print(f"   Cost: ${best['Cost ($)']:.4f}")
    print(f"   Validation Errors: {best['Validation Errors']}")
else:
    # Recommend highest confidence
    best = df.loc[df["Avg Confidence"].idxmax()]
    print(f"\n⚠️  No plan meets 0.8 confidence target")
    print(f"   Best Available: {best['Plan']}")
    print(f"   Average Confidence: {best['Avg Confidence']:.2f}")
    print(f"   Cost: ${best['Cost ($)']:.4f}")
    print(f"   Validation Errors: {best['Validation Errors']}")
